# Chapter 13: Heaps and Priority Queues

The programs from Chapter 13 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## A Tree Hidden in a List

`binary_heap.py`

In [1]:
"""A binary min-heap stored in a list.

The item at index i has its children at 2i + 1 and 2i + 2, and its
parent at (i - 1) // 2. Every parent is <= its children, so the
smallest item is always at index 0.
"""


class MinHeap:
    def __init__(self):
        self._items = []

    @classmethod
    def heapify(cls, items):
        """Build a heap from a list in linear time: sift down every
        parent, from the last one up to the root."""
        heap = cls()
        heap._items = list(items)
        for i in range(len(items) // 2 - 1, -1, -1):
            heap._sift_down(i)
        return heap

    def __len__(self):
        return len(self._items)

    def peek(self):
        if not self._items:
            raise IndexError("peek at an empty heap")
        return self._items[0]

    def push(self, value):
        self._items.append(value)  # add at the bottom, then move it up
        self._sift_up(len(self._items) - 1)

    def pop(self):
        """Remove and return the smallest item."""
        items = self._items
        if not items:
            raise IndexError("pop from an empty heap")
        last = items.pop()
        if not items:
            return last
        smallest, items[0] = items[0], last  # the last item to the top
        self._sift_down(0)
        return smallest

    def _sift_up(self, i):
        items = self._items
        while i > 0:
            parent = (i - 1) // 2
            if items[i] >= items[parent]:
                break
            items[i], items[parent] = items[parent], items[i]
            i = parent

    def _sift_down(self, i):
        items, n = self._items, len(self._items)
        while True:
            smallest = i
            for child in (2 * i + 1, 2 * i + 2):
                if child < n and items[child] < items[smallest]:
                    smallest = child
            if smallest == i:
                return
            items[i], items[smallest] = items[smallest], items[i]
            i = smallest

`heap_demo.py`

In [2]:
"""Watch the list behind the heap as items are pushed and popped."""

from binary_heap import MinHeap

h = MinHeap()
for x in [7, 4, 9, 1, 5]:
    h.push(x)
    print(f"push {x}: {h._items}")
while h:
    smallest = h.pop()
    print(f"pop -> {smallest}: {h._items}")

push 7: [7]
push 4: [4, 7]
push 9: [4, 7, 9]
push 1: [1, 4, 9, 7]
push 5: [1, 4, 9, 7, 5]
pop -> 1: [4, 5, 9, 7]
pop -> 4: [5, 7, 9]
pop -> 5: [7, 9]
pop -> 7: [9]
pop -> 9: []


## Building a Heap in Linear Time

`count_swaps.py`

In [3]:
"""Building a heap of n items: n pushes or one heapify? Count the swaps
on the worst input for pushing, a list in decreasing order."""

from binary_heap import MinHeap


class Counting(MinHeap):
    swaps = 0

    def _sift_up(self, i):
        items = self._items
        while i > 0 and items[i] < items[(i - 1) // 2]:
            parent = (i - 1) // 2
            items[i], items[parent] = items[parent], items[i]
            Counting.swaps += 1
            i = parent

    def _sift_down(self, i):
        items, n = self._items, len(self._items)
        while True:
            small = min(
                (c for c in (i, 2 * i + 1, 2 * i + 2) if c < n),
                key=items.__getitem__,
            )
            if small == i:
                return
            items[i], items[small] = items[small], items[i]
            Counting.swaps += 1
            i = small


print(f"{'n':>9} {'n pushes':>10} {'heapify':>9}")
for n in [1_000, 10_000, 100_000]:
    data = list(range(n, 0, -1))
    Counting.swaps = 0
    h = Counting()
    for x in data:
        h.push(x)
    pushes = Counting.swaps
    Counting.swaps = 0
    Counting.heapify(data)
    print(f"{n:>9,} {pushes:>10,} {Counting.swaps:>9,}")

        n   n pushes   heapify
    1,000      7,987       992
   10,000    113,631     9,992


  100,000  1,468,946    99,990


## Timing

`bench_heap.py`

In [4]:
"""Building a heap: n pushes against heapify, ours and heapq's."""

import heapq
import random

import bench
from binary_heap import MinHeap

rng = random.Random(2026)


def data(n):
    return (rng.sample(range(n), n),)


def ours_push(items):
    h = MinHeap()
    for x in items:
        h.push(x)


def heapq_push(items):
    h = []
    for x in items:
        heapq.heappush(h, x)


sizes = [10_000, 20_000, 40_000, 80_000]
results = {
    "MinHeap pushes": bench.doubling(ours_push, data, sizes),
    "MinHeap.heapify": bench.doubling(MinHeap.heapify, data, sizes),
    "heappush": bench.doubling(heapq_push, data, sizes),
    "heapq.heapify": bench.doubling(heapq.heapify, data, sizes, fresh=True),
}
bench.table(results)

         n  MinHeap pushes  MinHeap.heapify        heappush   heapq.heapify
    10,000         2.08 ms          1.82 ms          581 us          275 us
    20,000    4.18 ms x2.0     3.65 ms x2.0     1.2 ms x2.1     567 us x2.1
    40,000    10.2 ms x2.4     7.92 ms x2.2    2.74 ms x2.3     1.4 ms x2.5
    80,000    23.6 ms x2.3     18.4 ms x2.3    6.64 ms x2.4    3.75 ms x2.7


## The heapq Module: Four Patterns

`heap_patterns.py`

In [5]:
"""heapq, the standard library heap, in four common patterns."""

import heapq

scores = [88, 42, 97, 63, 71, 99, 54]

# 1. The k largest (or smallest) items
print(heapq.nlargest(3, scores), heapq.nsmallest(2, scores))

# 2. A priority queue of tasks; the counter breaks ties in arrival order
tasks, counter = [], 0
for priority, task in [(2, "email"), (1, "fix bug"), (2, "lunch")]:
    heapq.heappush(tasks, (priority, counter, task))
    counter += 1
print([heapq.heappop(tasks)[2] for _ in range(3)])

# 3. Merge sorted lists lazily, one item at a time
print(list(heapq.merge([1, 4, 7], [2, 5, 8], [3, 6, 9])))

# 4. A max-heap: store negated values
max_heap = [-s for s in scores]
heapq.heapify(max_heap)
print(-heapq.heappop(max_heap))

[99, 97, 88] [42, 54]
['fix bug', 'email', 'lunch']
[1, 2, 3, 4, 5, 6, 7, 8, 9]
99


## A Running Median

`running_median.py`

In [6]:
"""The median of a stream of numbers, after each new one, with two
heaps: a max-heap for the smaller half and a min-heap for the larger."""

import heapq


class RunningMedian:
    def __init__(self):
        self.low = []  # max-heap of the smaller half (values negated)
        self.high = []  # min-heap of the larger half

    def add(self, x):
        heapq.heappush(self.low, -x)
        # move the biggest of the low half up, then rebalance
        heapq.heappush(self.high, -heapq.heappop(self.low))
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2


m = RunningMedian()
for x in [5, 15, 1, 3, 8, 7]:
    m.add(x)
    print(f"add {x:>2}: median {m.median()}")

add  5: median 5
add 15: median 10.0
add  1: median 5
add  3: median 4.0
add  8: median 5
add  7: median 6.0
